Hier werden die Datensätze eingelesen und pro Datensatz ein erster Überblick gewonnen. 
Gelöscht werden nur eindeutig doppelte Zeilen oder eindeutig doppelte Variablen

In [10]:
import sys, mlflow, pandas as pd
print(sys.executable)
print(mlflow.__version__)
print(pd.__version__)
import os
from pathlib import Path

C:\Users\nelid\anaconda3\envs\store-sales-env\python.exe
3.9.0
2.3.3


In [11]:
BASE_DIR = Path(r"C:\Users\nelid\Documents\Kaggle Competitions\Store Sales Competition\store-sales")
DATA_DIR = BASE_DIR / "data" / "raw"

DATA_DIR, list(DATA_DIR.glob("*.csv"))

(WindowsPath('C:/Users/nelid/Documents/Kaggle Competitions/Store Sales Competition/store-sales/data/raw'),
 [WindowsPath('C:/Users/nelid/Documents/Kaggle Competitions/Store Sales Competition/store-sales/data/raw/holidays_events.csv'),
  WindowsPath('C:/Users/nelid/Documents/Kaggle Competitions/Store Sales Competition/store-sales/data/raw/oil.csv'),
  WindowsPath('C:/Users/nelid/Documents/Kaggle Competitions/Store Sales Competition/store-sales/data/raw/sample_submission.csv'),
  WindowsPath('C:/Users/nelid/Documents/Kaggle Competitions/Store Sales Competition/store-sales/data/raw/stores.csv'),
  WindowsPath('C:/Users/nelid/Documents/Kaggle Competitions/Store Sales Competition/store-sales/data/raw/test.csv'),
  WindowsPath('C:/Users/nelid/Documents/Kaggle Competitions/Store Sales Competition/store-sales/data/raw/train.csv'),
  WindowsPath('C:/Users/nelid/Documents/Kaggle Competitions/Store Sales Competition/store-sales/data/raw/transactions.csv')])

In [12]:
# Datensätze einlesen und shape darstellen

files = {
    "train": "train.csv",
    "test": "test.csv",
    "stores": "stores.csv",
    "oil": "oil.csv",
    "holidays_events": "holidays_events.csv",
    "transactions": "transactions.csv",
    "sample_submission": "sample_submission.csv",
}

dfs = {}
for name, fname in files.items():
    path = DATA_DIR / fname
    df = pd.read_csv(path)
    dfs[name] = df
    print(f"{name}: shape={df.shape}")


train: shape=(3000888, 6)
test: shape=(28512, 5)
stores: shape=(54, 5)
oil: shape=(1218, 2)
holidays_events: shape=(350, 6)
transactions: shape=(83488, 3)
sample_submission: shape=(28512, 2)


In [13]:
# Duplikate prüfen und entfernen

def clean_duplicates(df: pd.DataFrame, name: str) -> pd.DataFrame:
    print(f"\n=== {name} ===")
    print("Original shape:", df.shape)

    # Duplizierte Zeilen
    dup_rows = df.duplicated().sum()
    print("Duplicate rows:", dup_rows)
    if dup_rows > 0:
        df = df.drop_duplicates()
        print("Shape after dropping duplicate rows:", df.shape)

    # Duplizierte Spalten
    dup_cols = df.columns.duplicated()
    if dup_cols.any():
        print("Duplicate columns found:", list(df.columns[dup_cols]))
        df = df.loc[:, ~dup_cols]
        print("Shape after dropping duplicate columns:", df.shape)
    else:
        print("No duplicate columns.")

    return df

cleaned_dfs = {}
for name, df in dfs.items():
    cleaned_dfs[name] = clean_duplicates(df, name)



=== train ===
Original shape: (3000888, 6)
Duplicate rows: 0
No duplicate columns.

=== test ===
Original shape: (28512, 5)
Duplicate rows: 0
No duplicate columns.

=== stores ===
Original shape: (54, 5)
Duplicate rows: 0
No duplicate columns.

=== oil ===
Original shape: (1218, 2)
Duplicate rows: 0
No duplicate columns.

=== holidays_events ===
Original shape: (350, 6)
Duplicate rows: 0
No duplicate columns.

=== transactions ===
Original shape: (83488, 3)
Duplicate rows: 0
No duplicate columns.

=== sample_submission ===
Original shape: (28512, 2)
Duplicate rows: 0
No duplicate columns.


In [14]:
# Jetzt ersten Überblick pro Datensatz gewinnen. 

for name, df in cleaned_dfs.items():
    print(f"\n##### {name} #####")
    print(df.head())
    print("\nInfo:")
    print(df.info())
    print("\nDescribe (numeric):")
    display(df.describe())



##### train #####
   id        date  store_nbr      family  sales  onpromotion
0   0  2013-01-01          1  AUTOMOTIVE    0.0            0
1   1  2013-01-01          1   BABY CARE    0.0            0
2   2  2013-01-01          1      BEAUTY    0.0            0
3   3  2013-01-01          1   BEVERAGES    0.0            0
4   4  2013-01-01          1       BOOKS    0.0            0

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         object 
 2   store_nbr    int64  
 3   family       object 
 4   sales        float64
 5   onpromotion  int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 137.4+ MB
None

Describe (numeric):


,id,store_nbr,sales,onpromotion
count,3.000888e+06,3.000888e+06,3.000888e+06,3.000888e+06
mean,1.500444e+06,2.750000e+01,3.577757e+02,2.602770e+00
std,8.662819e+05,1.558579e+01,1.101998e+03,1.221888e+01
min,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00
25%,7.502218e+05,1.400000e+01,0.000000e+00,0.000000e+00
50%,1.500444e+06,2.750000e+01,1.100000e+01,0.000000e+00
75%,2.250665e+06,4.100000e+01,1.958473e+02,0.000000e+00
max,3.000887e+06,5.400000e+01,1.247170e+05,7.410000e+02



##### test #####
        id        date  store_nbr      family  onpromotion
0  3000888  2017-08-16          1  AUTOMOTIVE            0
1  3000889  2017-08-16          1   BABY CARE            0
2  3000890  2017-08-16          1      BEAUTY            2
3  3000891  2017-08-16          1   BEVERAGES           20
4  3000892  2017-08-16          1       BOOKS            0

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28512 entries, 0 to 28511
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           28512 non-null  int64 
 1   date         28512 non-null  object
 2   store_nbr    28512 non-null  int64 
 3   family       28512 non-null  object
 4   onpromotion  28512 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 1.1+ MB
None

Describe (numeric):


,id,store_nbr,onpromotion
count,2.851200e+04,28512.000000,28512.000000
mean,3.015144e+06,27.500000,6.965383
std,8.230850e+03,15.586057,20.683952
min,3.000888e+06,1.000000,0.000000
25%,3.008016e+06,14.000000,0.000000
50%,3.015144e+06,27.500000,0.000000
75%,3.022271e+06,41.000000,6.000000
max,3.029399e+06,54.000000,646.000000



##### stores #####
   store_nbr           city                           state type  cluster
0          1          Quito                       Pichincha    D       13
1          2          Quito                       Pichincha    D       13
2          3          Quito                       Pichincha    D        8
3          4          Quito                       Pichincha    D        9
4          5  Santo Domingo  Santo Domingo de los Tsachilas    D        4

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   store_nbr  54 non-null     int64 
 1   city       54 non-null     object
 2   state      54 non-null     object
 3   type       54 non-null     object
 4   cluster    54 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 2.2+ KB
None

Describe (numeric):


...

,date,type,locale,locale_name,description,transferred
count,350,350,350,350,350,350
unique,312,6,3,24,103,2
top,2014-06-25,Holiday,National,Ecuador,Carnaval,False
freq,4,221,174,174,10,338



##### transactions #####
         date  store_nbr  transactions
0  2013-01-01         25           770
1  2013-01-02          1          2111
2  2013-01-02          2          2358
3  2013-01-02          3          3487
4  2013-01-02          4          1922

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83488 entries, 0 to 83487
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   date          83488 non-null  object
 1   store_nbr     83488 non-null  int64 
 2   transactions  83488 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.9+ MB
None

Describe (numeric):


,store_nbr,transactions
count,83488.000000,83488.000000
mean,26.939237,1694.602158
std,15.608204,963.286644
min,1.000000,5.000000
25%,13.000000,1046.000000
50%,27.000000,1393.000000
75%,40.000000,2079.000000
max,54.000000,8359.000000



##### sample_submission #####
        id  sales
0  3000888    0.0
1  3000889    0.0
2  3000890    0.0
3  3000891    0.0
4  3000892    0.0

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28512 entries, 0 to 28511
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      28512 non-null  int64  
 1   sales   28512 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 445.6 KB
None

Describe (numeric):


,id,sales
count,2.851200e+04,28512.0
mean,3.015144e+06,0.0
std,8.230850e+03,0.0
min,3.000888e+06,0.0
25%,3.008016e+06,0.0
50%,3.015144e+06,0.0
75%,3.022271e+06,0.0
max,3.029399e+06,0.0


In [15]:
# "date" muss in allen Dateien zu datetime konvertiert werden

from pandas.api.types import is_datetime64_any_dtype as is_datetime

def convert_date_column(df: pd.DataFrame, name: str) -> pd.DataFrame:
    if "date" in df.columns:
        print(f"Converting 'date' in {name} from {df['date'].dtype} to datetime")
        df = df.copy()
        df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce")
        # Optional: prüfen, ob etwas NaT geworden ist
        n_nat = df["date"].isna().sum()
        if n_nat > 0:
            print(f"  Warning: {n_nat} rows in {name} have invalid dates (NaT).")
    else:
        print(f"{name}: no 'date' column.")
    return df

converted_dfs = {}
for name, df in cleaned_dfs.items():  
    converted_dfs[name] = convert_date_column(df, name)


Converting 'date' in train from object to datetime
Converting 'date' in test from object to datetime
stores: no 'date' column.
Converting 'date' in oil from object to datetime
Converting 'date' in holidays_events from object to datetime
Converting 'date' in transactions from object to datetime
sample_submission: no 'date' column.


In [16]:
# Check, ob nun date datetime ist:

for name, df in converted_dfs.items():
    if "date" in df.columns:
        print(name, type(df["date"].iloc[0]), df["date"].dtype)


train <class 'pandas._libs.tslibs.timestamps.Timestamp'> datetime64[ns]
test <class 'pandas._libs.tslibs.timestamps.Timestamp'> datetime64[ns]
oil <class 'pandas._libs.tslibs.timestamps.Timestamp'> datetime64[ns]
holidays_events <class 'pandas._libs.tslibs.timestamps.Timestamp'> datetime64[ns]
transactions <class 'pandas._libs.tslibs.timestamps.Timestamp'> datetime64[ns]


In [17]:
# Hier werden die Arbeitsdatensätze gespeichert
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

for name, df in converted_dfs.items():
    out_path = PROCESSED_DIR / f"{name}_processed.csv"
    df.to_csv(out_path, index=False)
    print(f"Saved {name} -> {out_path}")



Saved train -> C:\Users\nelid\Documents\Kaggle Competitions\Store Sales Competition\store-sales\data\processed\train_processed.csv
Saved test -> C:\Users\nelid\Documents\Kaggle Competitions\Store Sales Competition\store-sales\data\processed\test_processed.csv
Saved stores -> C:\Users\nelid\Documents\Kaggle Competitions\Store Sales Competition\store-sales\data\processed\stores_processed.csv
Saved oil -> C:\Users\nelid\Documents\Kaggle Competitions\Store Sales Competition\store-sales\data\processed\oil_processed.csv
Saved holidays_events -> C:\Users\nelid\Documents\Kaggle Competitions\Store Sales Competition\store-sales\data\processed\holidays_events_processed.csv
Saved transactions -> C:\Users\nelid\Documents\Kaggle Competitions\Store Sales Competition\store-sales\data\processed\transactions_processed.csv
Saved sample_submission -> C:\Users\nelid\Documents\Kaggle Competitions\Store Sales Competition\store-sales\data\processed\sample_submission_processed.csv
